# Notebook 01 — EDA and Preprocessing
**Project:** TruthLens — Multilingual Fake News Detection  
**Goal:** Load datasets, explore data, clean text, save processed files

---

## 1. Install & Import Libraries

In [ ]:
# Install if not already done
# !pip install pandas numpy matplotlib seaborn wordcloud nltk scikit-learn

import os
import re
import string
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from collections import Counter

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# Word cloud
try:
    from wordcloud import WordCloud
    WORDCLOUD = True
except ImportError:
    WORDCLOUD = False
    print('WordCloud not installed. Run: pip install wordcloud')

plt.style.use('dark_background')
sns.set_palette('husl')

print('✓ All libraries imported successfully')
print(f'Pandas: {pd.__version__}, NumPy: {np.__version__}')

## 2. Load Datasets

In [ ]:
DATASETS_DIR = '../datasets/'
MODELS_DIR   = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

dfs = []

# ── WELFake ──────────────────────────────────────────────────────────────────
welfake_path = os.path.join(DATASETS_DIR, 'welfake_dataset.csv')
if os.path.exists(welfake_path):
    wf = pd.read_csv(welfake_path)
    print(f'WELFake loaded: {wf.shape}')
    print(f'Columns: {list(wf.columns)}')
    # WELFake: columns are Unnamed:0, title, text, label (0=real, 1=fake)
    wf['text'] = wf['title'].fillna('') + ' ' + wf['text'].fillna('')
    wf = wf[['text', 'label']].copy()
    wf['source'] = 'welfake'
    dfs.append(wf)
    print(f'WELFake label distribution:\n{wf["label"].value_counts()}\n')
else:
    print(f'WELFake not found at {welfake_path}')

# ── ISOT ─────────────────────────────────────────────────────────────────────
isot_fake_path = os.path.join(DATASETS_DIR, 'isot_fake.csv')
isot_real_path = os.path.join(DATASETS_DIR, 'isot_real.csv')
if os.path.exists(isot_fake_path) and os.path.exists(isot_real_path):
    isot_fake = pd.read_csv(isot_fake_path)
    isot_real = pd.read_csv(isot_real_path)
    isot_fake['label'] = 1
    isot_real['label'] = 0
    isot = pd.concat([isot_fake, isot_real], ignore_index=True)
    isot['text'] = isot['title'].fillna('') + ' ' + isot['text'].fillna('')
    isot = isot[['text', 'label']].copy()
    isot['source'] = 'isot'
    dfs.append(isot)
    print(f'ISOT loaded: {isot.shape}')
    print(f'ISOT label distribution:\n{isot["label"].value_counts()}\n')
else:
    print('ISOT files not found — skipping')

# ── Combine all ──────────────────────────────────────────────────────────────
if len(dfs) == 0:
    raise FileNotFoundError('No datasets found! Place welfake_dataset.csv in datasets/ folder.')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'\nTotal combined dataset: {df_raw.shape}')
print(f'Label distribution:\n{df_raw["label"].value_counts()}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Fake News Dataset — EDA Overview', fontsize=16, fontweight='bold', y=1.02)

# 1. Class distribution
label_counts = df_raw['label'].value_counts()
colors = ['#3ecf8e', '#ff6b6b']
axes[0].pie(label_counts.values, labels=['Real', 'Fake'], autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Class Distribution', fontsize=13)

# 2. Article length distribution
df_raw['word_count'] = df_raw['text'].fillna('').apply(lambda x: len(str(x).split()))
df_fake = df_raw[df_raw['label'] == 1]['word_count'].clip(0, 1000)
df_real = df_raw[df_raw['label'] == 0]['word_count'].clip(0, 1000)
axes[1].hist(df_real, bins=50, alpha=0.7, color='#3ecf8e', label='Real', density=True)
axes[1].hist(df_fake, bins=50, alpha=0.7, color='#ff6b6b', label='Fake', density=True)
axes[1].set_title('Article Length Distribution', fontsize=13)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Density')
axes[1].legend()

# 3. Source distribution (if multiple datasets)
if 'source' in df_raw.columns:
    source_counts = df_raw['source'].value_counts()
    axes[2].bar(source_counts.index, source_counts.values,
                color=['#7c6af7', '#3ecf8e', '#f7b731'][:len(source_counts)])
    axes[2].set_title('Articles per Dataset', fontsize=13)
    axes[2].set_xlabel('Dataset')
    axes[2].set_ylabel('Count')
    for i, v in enumerate(source_counts.values):
        axes[2].text(i, v + 200, f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../models/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA chart saved to models/eda_overview.png')

In [ ]:
# ── Top words in FAKE vs REAL ─────────────────────────────────────────────────
stop_words = set(stopwords.words('english'))

def get_top_words(texts, n=30):
    all_words = []
    for text in texts:
        words = str(text).lower().split()
        words = [w.strip(string.punctuation) for w in words
                 if w not in stop_words and len(w) > 3 and w.isalpha()]
        all_words.extend(words)
    return Counter(all_words).most_common(n)

fake_texts = df_raw[df_raw['label'] == 1]['text'].fillna('').tolist()[:5000]
real_texts = df_raw[df_raw['label'] == 0]['text'].fillna('').tolist()[:5000]

top_fake = get_top_words(fake_texts)
top_real = get_top_words(real_texts)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Fake top words
fake_words, fake_counts = zip(*top_fake[:20])
axes[0].barh(fake_words[::-1], fake_counts[::-1], color='#ff6b6b')
axes[0].set_title('Top 20 Words in FAKE News', fontsize=13)
axes[0].set_xlabel('Frequency')

# Real top words
real_words, real_counts = zip(*top_real[:20])
axes[1].barh(real_words[::-1], real_counts[::-1], color='#3ecf8e')
axes[1].set_title('Top 20 Words in REAL News', fontsize=13)
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../models/top_words.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Word Clouds ───────────────────────────────────────────────────────────────
if WORDCLOUD:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    fake_text_joined = ' '.join(fake_texts[:2000])
    real_text_joined = ' '.join(real_texts[:2000])

    wc_fake = WordCloud(width=800, height=400, background_color='black',
                        colormap='Reds', max_words=100,
                        stopwords=stop_words).generate(fake_text_joined)
    wc_real = WordCloud(width=800, height=400, background_color='black',
                        colormap='Greens', max_words=100,
                        stopwords=stop_words).generate(real_text_joined)

    axes[0].imshow(wc_fake, interpolation='bilinear')
    axes[0].axis('off')
    axes[0].set_title('FAKE News Word Cloud', fontsize=14, color='#ff6b6b')

    axes[1].imshow(wc_real, interpolation='bilinear')
    axes[1].axis('off')
    axes[1].set_title('REAL News Word Cloud', fontsize=14, color='#3ecf8e')

    plt.tight_layout()
    plt.savefig('../models/wordclouds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Word clouds saved!')
else:
    print('Install wordcloud: pip install wordcloud')

## 4. Text Preprocessing & Cleaning

In [ ]:
stemmer = PorterStemmer()

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation + string.digits))
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [stemmer.stem(w) for w in tokens
              if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

print('Cleaning text... (this takes 2-3 minutes for large datasets)')

# Use tqdm for progress bar if available
try:
    from tqdm import tqdm
    tqdm.pandas()
    df_raw['cleaned_text'] = df_raw['text'].progress_apply(clean_text)
except ImportError:
    df_raw['cleaned_text'] = df_raw['text'].apply(clean_text)

# Drop rows where cleaned text is too short
df_clean = df_raw[df_raw['cleaned_text'].str.len() > 20].copy()
df_clean = df_clean.dropna(subset=['cleaned_text', 'label'])
df_clean['label'] = df_clean['label'].astype(int)

print(f'\nBefore cleaning: {len(df_raw):,} rows')
print(f'After cleaning:  {len(df_clean):,} rows')
print(f'Removed:         {len(df_raw) - len(df_clean):,} rows')
print(f'\nLabel distribution after cleaning:')
print(df_clean['label'].value_counts())

In [ ]:
# ── Show a sample before vs after cleaning ────────────────────────────────────
sample_idx = df_clean.index[0]
print('BEFORE cleaning:')
print(df_clean.loc[sample_idx, 'text'][:300])
print('\nAFTER cleaning:')
print(df_clean.loc[sample_idx, 'cleaned_text'][:300])
print(f'\nLabel: {"FAKE" if df_clean.loc[sample_idx, "label"] == 1 else "REAL"}')

## 5. Save Processed Dataset

In [ ]:
# Save processed dataset for use in next notebooks
save_path = '../datasets/processed_dataset.csv'
df_clean[['cleaned_text', 'label']].to_csv(save_path, index=False)
print(f'Processed dataset saved → {save_path}')
print(f'Shape: {df_clean[["cleaned_text", "label"]].shape}')

# Summary statistics
print('\n' + '='*50)
print('NOTEBOOK 01 COMPLETE — SUMMARY')
print('='*50)
print(f'Total articles:  {len(df_clean):,}')
print(f'Fake articles:   {(df_clean["label"]==1).sum():,}')
print(f'Real articles:   {(df_clean["label"]==0).sum():,}')
print(f'Avg text length: {df_clean["cleaned_text"].str.split().apply(len).mean():.0f} words')
print('\nNext step → Run Notebook 02: Supervised Models')